![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 03 | Evaluate — the Report

*Project 3 — AI Product: From Model to Product*

---

**This notebook is your report.** It is read on its own, by someone who has not seen notebooks 01 and 02: the question, the evidence, what you concluded and how far to trust it.

> **Done when:** the final version is evaluated once, its failures are analysed with examples, its cost is known, and the conclusions answer the question you framed in notebook 01.

---
## 0. Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *
import textwrap

pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

W1111 14:53:24.529000 25761 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
config, collection, embedder, client = load_pipeline()
eval_set = pd.read_csv(ROOT / "data" / "eval_questions.csv", dtype={"source_review_ids": str})
RESULTS = ROOT / "data" / "results"
baseline_graded = pd.read_csv(RESULTS / "baseline_llm_only_graded.csv")
print(f"{collection.count():,} chunks | {len(eval_set)} questions | baseline answers: {len(baseline_graded)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

44,477 chunks | 30 questions | baseline answers: 30


---
## 1. The question

The user, the decision, the metric and the baseline score, in a few sentences, as framed in notebook 01.

**The user:** a PC player deciding whether to buy a game, or wondering how it runs and plays today. Steam shows one score and thousands of unsorted reviews.

**The product:** an assistant that answers questions about 23 games from 22,707 English Steam reviews (collected on 10 November 2026), cites the reviews it used, and says "I don't know" when they do not answer. It retrieves the 5 most relevant review chunks for the selected game (Chroma, `all-MiniLM-L6-v2` embeddings) and answers from them with `gpt-5.4-nano`.

**The decision:** buy now, wait for a patch or a sale, or skip the game.

**The metric, and which mistake costs most.** A confident wrong answer is the worst mistake (the player spends money on something invented), an answer without a source is next, and a refusal is the cheapest. Four things are measured on a 30-question evaluation set written before building the system: the **retrieval hit rate**, **answer correctness** (graded with a rubric fixed in advance), **citation accuracy**, and whether the 4 **out-of-scope** questions are refused.

**The baseline to beat:** the same LLM without any reviews scores **0.33** on the 26 answerable questions, refuses 12 of them, handles 3 of the 4 out-of-scope questions, and **never hallucinates**. For retrieval, 5 chunks picked at random would hit 52% of the time.

---
## 2. The final evaluation

- **Model track:** the final model, evaluated **once** on the test set, compared with its validation score and with the baseline.
- **LLM track:** the whole evaluation set. Report separately: the retrieval hit rate (or the share of questions with the right tools), whether answers are correct and cite the right source, and whether the out-of-scope questions are refused.

In [3]:
rag_file = RESULTS / "rag_final.csv"
if rag_file.exists():
    rag = pd.read_csv(rag_file, dtype={"cited_ids": str, "retrieved_ids": str, "invented_citations": str})
    print("Loaded the saved answers (graded): not calling the API again")
else:
    rows = []
    for q in eval_set.itertuples():
        out = answer_question(q.question, collection, embedder, client, k=config["k"],
                              game=q.game if config["game_filter"] else None,
                              min_playtime=q.min_playtime, steam_deck=q.steam_deck)
        rows.append({"id": q.id, "type": q.type, "question": q.question, **out})
        print(f"{q.id:4s} {out['input_tokens']:5d} in  {out['output_tokens']:4d} out  {out['seconds']:4.1f} s")
    rag = pd.DataFrame(rows)
    rag.to_csv(rag_file, index=False)

print(f"\nInvented citations: {rag['invented_citations'].fillna('').astype(bool).sum()} of {len(rag)} answers")

F1    1114 in   156 out   2.5 s
F2     762 in   145 out   1.9 s
F3     732 in   117 out   1.6 s
F4     966 in   176 out   1.9 s
F5     889 in   182 out   2.0 s
F6    1015 in   193 out   2.3 s
F7    1013 in   136 out   1.7 s
F8     837 in   197 out   2.0 s
F9     819 in   185 out   1.9 s
F10   1012 in   173 out   1.8 s
F11    737 in   168 out   2.1 s
T1     854 in   189 out   1.9 s
T2     826 in   151 out   1.8 s
T3    1004 in   299 out   2.5 s
T4     961 in   149 out   1.6 s
P1     958 in   184 out   1.8 s
P2    1114 in   156 out   1.6 s
C1     857 in   215 out   2.5 s
C2    1014 in   208 out   2.0 s
C3     873 in   222 out   2.1 s
E1     848 in   158 out   1.8 s
E2     858 in   205 out   2.2 s
R1     735 in   186 out   2.1 s
R2     994 in   160 out   2.0 s
H1    1020 in   153 out   2.0 s
H2     931 in   136 out   1.7 s
O1     661 in   104 out   1.5 s
O2     792 in    86 out   1.5 s
O3     782 in    84 out   1.2 s
O4    1018 in    87 out   1.2 s

Invented citations: 0 of 30 answers


In [4]:
to_grade = (rag.merge(eval_set[["id", "expected_answer"]], on="id")
               .merge(baseline_graded[["id", "answer", "score"]].rename(
                      columns={"answer": "baseline_answer", "score": "baseline_score"}), on="id"))

with open(RESULTS / "rag_for_grading.md", "w") as f:
    for r in to_grade.itertuples():
        f.write(f"## {r.id} ({r.type}): {r.question}\n\n")
        f.write(f"**Expected:** {r.expected_answer}\n\n")
        f.write(f"**RAG:** {r.answer}\n\n")
        f.write(f"*Cited: {r.cited_ids} | invented: {r.invented_citations} | closest distance: {r.min_distance}*\n\n")
        f.write(f"**Baseline (score {r.baseline_score}):** {r.baseline_answer}\n\n---\n\n")
print("Written to data/results/rag_for_grading.md")

Written to data/results/rag_for_grading.md


In [5]:
rag_grades = {
    # id:   (score, refused, hallucinated, note)
    "F1":  (0.5, False, False, "Strategic and dynamic; misses the dice frustration"),
    "F2":  (0.5, False, False, "Yes, with non-fans' view; misses God Mode, Hell Mode, caveats"),
    "F3":  (0.5, False, False, "Yes but thin: no reasons (open world, quests, sale); baseline was fuller"),
    "F4":  (1,   False, False, "Insane value, ~$15, 50+ hours, criminally underpriced"),
    "F5":  (1,   False, False, "Strong add-on, story, missions, characters; misses the 2.0 link"),
    "F6":  (0.5, False, False, "'Mixed' without landing the answer: for most, the pace is part of the appeal"),
    "F7":  (0.5, False, False, "Mods optional; misses the bug-fix minority"),
    "F8":  (0.5, False, False, "Describes the parry system more than players' verdict"),
    "F9":  (1,   False, False, "Worth it, 4-player chaos, glitchy co-op caveat"),
    "F10": (0.5, False, False, "Runs well; misses the Steam Deck problems"),
    "F11": (0.5, False, False, "Mixed, story okay-ish; misses villain and formula"),
    "T1":  (0.5, False, False, "Improved but mixed; uses 2023 reviews as 'still' problems"),
    "T2":  (1,   False, False, "Still stutters, freezes, drops, runs poorly after a long time"),
    "T3":  (1,   False, False, "Unpolished launch, responsive devs and updates, criticism remains"),
    "T4":  (0.5, False, False, "Frames it as general performance problems, not mainly mods"),
    "P1":  (0,   True,  False, "Honest refusal: no late-game chunks retrieved"),
    "P2":  (1,   False, False, "Perfectly optimized, no lag, multiplayer compatibility note"),
    "C1":  (0.5, False, False, "Favourable, Sekiro + Bloodborne; misses the critics"),
    "C2":  (0.5, False, False, "Not clearly better; misses 'depends on when'; cites a 0-hour review"),
    "C3":  (1,   False, False, "Ghost better story, Shadows praised for stealth and combat"),
    "E1":  (1,   False, False, "Difficulty, walk back, guidance, backtracking; says evidence is limited"),
    "E2":  (0.5, False, False, "Short story, dialogue; misses the MJ/Miles missions"),
    "R1":  (0.5, False, False, "Solid, stealth, linear, AI; misses Bond feel and gadgets"),
    "R2":  (0.5, False, False, "Leads with anticipation as evidence, finds seafaring, then hedges"),
    "H1":  (0.5, False, False, "Story worth it, port issues don't harm it; no negative reviews retrieved"),
    "H2":  (0.5, False, False, "Punishing, not a good introduction, hard but fair; thin"),
    "O1":  (1,   True,  False, "Refuses: reviews are PC only"),
    "O2":  (1,   True,  False, "Refuses: no dates in reviews"),
    "O3":  (1,   True,  False, "Refuses: reviews are about other games"),
    "O4":  (1,   True,  False, "Refuses: no current price"),
}

grade_table = pd.DataFrame.from_dict(rag_grades, orient="index",
                                     columns=["score", "refused", "hallucinated", "note"])
grade_table.index.name = "id"
rag_graded = rag.merge(grade_table.reset_index(), on="id")
rag_graded.to_csv(RESULTS / "rag_final_graded.csv", index=False)
print(f"Saved {len(rag_graded)} graded answers")

Saved 30 graded answers


In [6]:
both = rag_graded[["id", "type", "score", "refused", "hallucinated"]].merge(
    baseline_graded[["id", "score", "refused", "hallucinated"]], on="id", suffixes=("_rag", "_baseline"))
answerable = both["id"].isin(eval_set.loc[eval_set["answerable"], "id"])

summary = pd.DataFrame({
    system: [both.loc[answerable, f"score_{system}"].mean().round(2),
             int(both.loc[answerable, f"refused_{system}"].sum()),
             int(both.loc[~answerable, f"score_{system}"].sum()),
             int(both[f"hallucinated_{system}"].sum())]
    for system in ("rag", "baseline")},
    index=["Answerable score (of 1)", "Refused (of 26)", "Out of scope correct (of 4)", "Hallucinations"])
print(summary, "\n")

change = (both.loc[answerable, "score_rag"] - both.loc[answerable, "score_baseline"])
print(f"Better: {(change > 0).sum()}  |  same: {(change == 0).sum()}  |  worse: {(change < 0).sum()}\n")
both[answerable].groupby("type")[["score_rag", "score_baseline"]].mean().round(2).sort_values("score_rag", ascending=False)

                              rag  baseline
Answerable score (of 1)      0.63      0.33
Refused (of 26)              1.00     12.00
Out of scope correct (of 4)  4.00      3.00
Hallucinations               0.00      0.00 

Better: 14  |  same: 8  |  worse: 4



,score_rag,score_baseline
type,,
scarce_evidence,0.75,0.00
then_vs_now,0.75,0.00
comparison,0.67,0.33
fact,0.64,0.55
filtered,0.50,0.25
hard_case,0.50,0.50
recent,0.50,0.00


In [7]:
retrieval = evaluate_retrieval(eval_set, collection, embedder, k=config["k"],
                               use_game_filter=config["game_filter"])

sources = eval_set.set_index("id")["source_review_ids"]
answerable_rag = rag_graded[rag_graded["id"].isin(eval_set.loc[eval_set["answerable"], "id"])]
cites_a_source = [bool(set(str(r.cited_ids).split(";")) & set(str(sources[r.id]).split(";")))
                  for r in answerable_rag.itertuples()]

print(f"Retrieval hit rate:            {retrieval['hit'].mean():.2f}   (5 random chunks: 0.52)")
print(f"Answers citing a hand-picked source: {sum(cites_a_source)} of {len(cites_a_source)}")
print(f"Invented citations:            {rag['invented_citations'].fillna('').astype(bool).sum()} of {len(rag)}")

Retrieval hit rate:            0.85   (5 random chunks: 0.52)
Answers citing a hand-picked source: 4 of 26
Invented citations:            0 of 30


**The final evaluation, run once and graded with the rubric fixed in notebook 01.**

| | RAG system | LLM alone (baseline) |
|---|---|---|
| **Answerable score** (26 questions) | **0.63** | 0.33 |
| Correct / partial / wrong / refused | 8 / 17 / 0 / 1 | 4 / 9 / 1 / 12 |
| Out-of-scope questions refused correctly | **4 of 4** | 3 of 4 |
| Hallucinations | **0** | 0 |
| Retrieval hit rate (k = 5) | 0.85 (random: 0.52) | — |
| Invented citations | **0 of 30** | — |
| Answers citing a hand-picked source | 4 of 26 | — |

| Type | RAG | Baseline |
|---|---|---|
| scarce_evidence | **0.75** | 0.00 |
| then_vs_now | **0.75** | 0.00 |
| comparison | 0.67 | 0.33 |
| fact | 0.64 | 0.55 |
| filtered | 0.50 | 0.25 |
| hard_case | 0.50 | 0.50 |
| recent | **0.50** | 0.00 |

**The RAG system nearly doubles the baseline's score without giving up its honesty:** zero hallucinations for both, and no citation points to a review that was not retrieved. Reading every answer confirms that each cited review supports the claim it is cited for. It answers 25 of the 26 questions where the baseline refused 12, and it handles all four out-of-scope questions, including Stardew Valley, which the baseline answered from its own knowledge.

**The gains are largest where the LLM alone has no knowledge:** changes over time and recent games (0.00 → 0.75 and 0.50), and complaints about specific games (0.00 → 0.75). On famous older games, the gain is small (fact: 0.55 → 0.64).

**Question by question: 14 better, 8 the same, 4 worse.** Three regressions are revealing: on F3 (Hogwarts Legacy), F6 (Red Dead Redemption 2's pace) and F8 (Sekiro's combat), the baseline's broad knowledge gave fuller answers than 5 retrieved chunks could. The fourth, P1 (Elden Ring's late game), is an honest refusal: retrieval found no chunk about the late game, and the system said so instead of guessing.

**How far to trust these numbers:** 26 answerable questions, each graded by one person, and one run per question. The same question can get different answers on two runs (notebook 02, R2), and each question is worth about 4 points, so differences of a few points are not reliable, and per-type scores rest on 2–11 questions.

---
## 3. Where it goes wrong

- **Model track:** a confusion matrix, per-class precision and recall, and a sample of mistakes **read in full**, with your reading of why.
- **LLM track:** every failure classified as a retrieval (or tool) failure or a generation failure, with examples.

Then: where should your product **not** be trusted?

In [13]:
questions = eval_set.set_index("id")
graded = rag_graded.set_index("id")

missing_evidence = {          # the evidence each partial answer missed, from the grading notes
    "F1": r"\bmiss(?:ed|es|ing)?\b|\bdice\b|\brng\b|\d+ ?%",
    "F2": r"god mode|hell mode|repetitive",
    "F3": r"open world|quests?|on sale|explor",
    "F6": r"part of|immers|charm|deliberate|reward",
    "F7": r"\bbugs?\b|unofficial patch|interface|\bui\b",
    "F8": r"\bbest\b|satisf|incredible|masterpiece",
    "F10": r"steam deck|crash|freez",
    "F11": r"castillo|villain|esposito|formula|same old",
    "T4": r"f4se|script extender|\bmods?\b",
    "P1": r"late.?game|end.?game|malenia|maliketh|radagon|godfrey|mohg|farum|haligtree",
    "C1": r"too close|annoying|dodg|derivative|copy",
    "C2": r"2\.0|phantom liberty|after (?:the )?(?:update|patch)",
    "E2": r"mary jane|\bmj\b|\bmiles\b",
    "R1": r"\bbond\b|gadget|q.?watch",
    "H1": r"linear|premise|familiar",
    "H2": r"elden ring|build|starting class|curse|learning curve",
}

rows = []
for qid, pattern in missing_evidence.items():
    q = questions.loc[qid]
    hits = retrieve(q["question"], collection, embedder, k=config["k"],
                    game=q["game"] if config["game_filter"] else None,
                    min_playtime=q["min_playtime"], steam_deck=q["steam_deck"])
    found = hits["text"].str.contains(pattern, case=False, regex=True)
    rows.append({"id": qid, "evidence_in_chunks": int(found.sum()),
                 "failure": "generation" if found.any() else "retrieval",
                 "note": graded.loc[qid, "note"]})

rows += [
    {"id": "T1", "evidence_in_chunks": None, "failure": "generation",
     "note": "Time reasoning: uses 2023 reviews as evidence that problems 'still' exist"},
    {"id": "R2", "evidence_in_chunks": None, "failure": "generation",
     "note": "Leads with anticipation ('can't wait for the DLC') as evidence of reception"},
]
failures = pd.DataFrame(rows).sort_values(["failure", "id"])
print(failures["failure"].value_counts(), "\n")
failures

failure
generation    10
retrieval      8
Name: count, dtype: int64 



,id,evidence_in_chunks,failure,note
10,C1,1.0,generation,"Favourable, Sekiro + Bloodborne; misses the critics"
0,F1,1.0,generation,Strategic and dynamic; misses the dice frustration
6,F10,1.0,generation,Runs well; misses the Steam Deck problems
2,F3,1.0,generation,"Yes but thin: no reasons (open world, quests, sale); baseline was fuller"
3,F6,1.0,generation,"'Mixed' without landing the answer: for most, the pace is part of the appeal"
5,F8,1.0,generation,Describes the parry system more than players' verdict
13,R1,2.0,generation,"Solid, stealth, linear, AI; misses Bond feel and gadgets"
17,R2,NaN,generation,Leads with anticipation ('can't wait for the DLC') as evidence of reception
16,T1,NaN,generation,Time reasoning: uses 2023 reviews as evidence that problems 'still' exist
8,T4,2.0,generation,"Frames it as general performance problems, not mainly mods"


In [14]:
qid = "F8"   # change to inspect any question
q = questions.loc[qid]
for h in retrieve(q["question"], collection, embedder, k=config["k"], game=q["game"]).itertuples():
    print(f"[{h.review_id}] {textwrap.fill(h.text, 110)}\n")

[204372242] Sekiro is a game where you parry. its a deceptively simple concept around which one of the best if not the
best combat systems in gaming is built.

[198944956] See, what got me confused is that Sekiro kept getting described as a "parry game". And while parrying is
somewhat of an important mechanic in this game, Sekiro should be more accurately described as an "aggression
game". This is because that unlike other parry focused games out there, Sekiro isn't really about the patient
dance of parrying and counterattacking. Really, it's more about hacking away at enemies and consistently
applying pressure due to the posture mechanic. Because guess what, it doesn't matter if your enemies block
your attacks; your goal for 95% of encounters is to overwhelm your enemies with enough attacks to allow you to
land that insta-kill deathblow.

[49683332] If you dislike using parry timings in combat—pressing the block button just as an enemy attack lands—then
Sekiro might not be for you.

[

In [15]:
for qid in ["F6", "F3", "F10", "F1", "C1"]:
    q = questions.loc[qid]
    hits = retrieve(q["question"], collection, embedder, k=config["k"], game=q["game"])
    matched = hits[hits["text"].str.contains(missing_evidence[qid], case=False, regex=True)]
    print(f"========== {qid}: {graded.loc[qid, 'note']}")
    for h in matched.itertuples():
        m = re.search(missing_evidence[qid], h.text, flags=re.IGNORECASE)
        start = max(m.start() - 150, 0)
        print(f"[{h.review_id}] ...{h.text[start:m.end() + 150]}...\n")

========== F6: 'Mixed' without landing the answer: for most, the pace is part of the appeal
[183212044] ...to detail set a new standard for open-world games. While the slow pacing and clunky mechanics may deter some players, those who stick with it will be rewarded with one of the most immersive and emotionally rich experiences in gaming.
Pros:
Stunning, immersive open world.
A powerful and emotional story wit...

========== F3: Yes but thin: no reasons (open world, quests, sale); baseline was fuller
[184098692] ... fan due to it's true to the movies, attention to detail etc - Gameplay mechanics wise though... Really needs reigning in a bit when it comes to the 'open world' side of things, overcooked on the collectables they become boring and chore-like... but, granted they are optional... and finally fights could be mo...

========== F10: Runs well; misses the Steam Deck problems
[108532089] ... fires on all cylinders and elevates this already great game to further greatness. I experi

In [22]:
# Read in full above: these matches were false positives, so the evidence was not really in the chunks
reclassify = {
    "F3":  "False positive: the matching chunk is a fan's view, not non-fans' reasons",
    "F10": "False positive: the chunk says the game 'never crashed' (negation)",
    "F1":  "False positive: the luck is in dialogue skill checks, not combat dice",
    "C1":  "False positive: 'instead of just copying it' is praise, not criticism",
}
for qid, reason in reclassify.items():
    failures.loc[failures["id"] == qid, ["failure", "note"]] = ["retrieval", reason]

failures.to_csv(RESULTS / "failure_analysis.csv", index=False)
print(failures["failure"].value_counts())
failures.sort_values(["failure", "id"])

failure
retrieval     12
generation     6
Name: count, dtype: int64


,id,evidence_in_chunks,failure,note
3,F6,1.0,generation,"'Mixed' without landing the answer: for most, the pace is part of the appeal"
5,F8,1.0,generation,Describes the parry system more than players' verdict
13,R1,2.0,generation,"Solid, stealth, linear, AI; misses Bond feel and gadgets"
17,R2,NaN,generation,Leads with anticipation ('can't wait for the DLC') as evidence of reception
16,T1,NaN,generation,Time reasoning: uses 2023 reviews as evidence that problems 'still' exist
8,T4,2.0,generation,"Frames it as general performance problems, not mainly mods"
10,C1,1.0,retrieval,"False positive: 'instead of just copying it' is praise, not criticism"
11,C2,0.0,retrieval,Not clearly better; misses 'depends on when'; cites a 0-hour review
12,E2,0.0,retrieval,"Short story, dialogue; misses the MJ/Miles missions"
0,F1,1.0,retrieval,"False positive: the luck is in dialogue skill checks, not combat dice"


**Every imperfect answer, classified.** 18 of the 26 answerable questions did not score 1 (17 partial, 1 refusal). For each, the grading note says what was missing; the question is whether that evidence was among the 5 retrieved chunks.
- **Retrieval failure (12):** the evidence was not retrieved, so the LLM could not have used it.
- **Generation failure (6):** the evidence was in the chunks, but the answer missed or misused it.

**How the classification was made, and corrected.** A pattern describing the missing evidence was checked against the retrieved chunks. This first pass gave 10 generation and 8 retrieval failures, but reading the five weakest matches in full showed that four were false positives: "the game **never crashed**" (a negation), "instead of just **copying** it" (praise), luck in **dialogue** skill checks (not combat dice), and a **fan's** view (not a non-fan's). Corrected, the split is 12 to 6. Like the hit rate, a keyword check needs reading to be trusted.

**Retrieval failures: the answer can only be as complete as the 5 chunks.**
- **Minority views and caveats are crowded out.** Skyrim's bug-fix minority (F7), God of War's Steam Deck problems (F10), the critics of Lies of P (C1): a few dissenting reviews rarely make the top 5 among hundreds of positive ones.
- **Embeddings find the topic, not the angle or the sentiment.** "Complaints about Spider-Man" retrieves positive overviews, so the Mary Jane and Miles missions are missed (E2). "Elden Ring's late game" retrieves general praise from long-time players, so the system refuses (P1). "Even with the bad PC reviews" retrieves no negative reviews (H1).
- **Time-dependent evidence is not targeted.** C2's answer depends on when players played, but no post-2.0 verdict was retrieved.

**Generation failures: the evidence was there, and the reasoning failed.**
- **Anticipation used as evidence (R2):** "players mention the DLC positively, saying they can't wait for it," while the only real evidence was one review about ships and underwater exploration.
- **Time reasoning (T1):** reviews from 2023 are presented as evidence that problems "still" exist.
- **Under-using the evidence (F8, F6):** F8 cites a review calling Sekiro's combat "one of the best if not the best combat systems in gaming," yet describes the mechanics instead of the verdict. F6 had a review saying the slow pace rewards those who stick with it, yet answered "mixed" without weighing it.
- **Framing (T4):** Fallout 4's update is presented as general performance problems, when the retrieved reviews tie them mainly to mods.
- **R1 (007 First Light), not verified by reading:** two retrieved chunks mention Bond, but "Bond" appears in almost every review of this game, so this classification may be a false positive like the four above.

**Where the product should not be trusted:**
1. **"Is there anything wrong with this game?"** Minority complaints are often missing, especially for well-liked games. An answer with no complaints is not evidence that there are none.
2. **Specific angles** (late game, a particular mode, complaints about a popular game): retrieval may return general opinions instead.
3. **Brand-new content:** names the embedding model does not know (a new DLC) are hard to retrieve, and anticipation can be mistaken for reception.
4. **"Is it still…?" questions:** the model can misread when reviews were written.
5. **Comparisons:** with the game filter, the other game is seen only through the selected game's reviews.
6. **Any single answer:** the same question can get different answers on two runs (notebook 02, R2).

**Where it can be trusted:** it never invented a source or a fact in 30 answers, it refused all four out-of-scope questions, and when evidence was missing it usually said so (P1, H2) instead of filling the gap.

---
## 4. Cost and speed

- **Model track:** training time, model size on disk, time to predict one text.
- **LLM track:** tokens, cost and time per question, and the total for the evaluation. What would 1,000 questions a day cost per month?

In [17]:
llm_costs = pd.DataFrame({
    name: [df["input_tokens"].mean(), df["output_tokens"].mean(),
           df["cost_usd"].mean(), df["seconds"].mean(), df["cost_usd"].sum()]
    for name, df in [("RAG", rag), ("Baseline (LLM alone)", baseline_graded)]
}, index=["Input tokens per question", "Output tokens per question",
          "Cost per question (USD)", "LLM time per question (s)", "Total for 30 questions (USD)"])
llm_costs.round(5)

,RAG,Baseline (LLM alone)
Input tokens per question,899.86667,81.76667
Output tokens per question,165.33333,104.13333
Cost per question (USD),0.00039,0.00015
LLM time per question (s),1.89300,1.79700
Total for 30 questions (USD),0.01160,0.00440


In [18]:
start = time.perf_counter()
for q in eval_set.itertuples():
    retrieve(q.question, collection, embedder, k=config["k"], game=q.game)
retrieval_s = (time.perf_counter() - start) / len(eval_set)

index_mb = sum(f.stat().st_size for f in (ROOT / config["index_path"]).rglob("*") if f.is_file()) / 1e6
print(f"Retrieval time per question: {retrieval_s * 1000:.0f} ms (local, free)")
print(f"Index on disk: {index_mb:.0f} MB for {collection.count():,} chunks")

Retrieval time per question: 16 ms (local, free)
Index on disk: 309 MB for 44,477 chunks


In [19]:
per_month = 1_000 * 30
monthly = rag["cost_usd"].mean() * per_month
print(f"1,000 questions a day → {per_month:,} a month → ${monthly:.2f} a month in LLM costs")
print(f"Total time per question: about {rag['seconds'].mean() + retrieval_s:.1f} s")

1,000 questions a day → 30,000 a month → $11.60 a month in LLM costs
Total time per question: about 1.9 s


**Cost and speed per question:**

| | RAG system | LLM alone |
|---|---|---|
| Input tokens | 900 | 82 |
| Output tokens | 165 | 104 |
| Cost | $0.00039 | $0.00015 |
| Time | 1.89 s (LLM) + 16 ms (retrieval) | 1.80 s |

**The cost of grounding is small.** Each RAG question sends about 11 times more input tokens, the 5 retrieved chunks, but costs only about 2.6 times more ($0.00039 against $0.00015). The reason is the price structure of `gpt-5.4-nano`: output tokens cost over six times more than input tokens ($1.25 against $0.20 per million), and the RAG answers are only about 60% longer. The whole evaluation (30 questions) cost $0.0116, against $0.0044 for the baseline.

**Retrieval adds almost no time.** Embedding the question and searching the index takes 16 ms per question, locally and at no cost, so a full answer takes about 1.9 seconds, almost entirely the LLM call. The index takes 309 MB on disk for 44,477 chunks, small enough to ship with the app.

**At 1,000 questions a day** (30,000 a month), the LLM would cost about **$11.60 a month**. Cost is not a constraint for this product; answer quality and retrieval are. A larger model would likely write fuller answers from the same chunks, but 12 of the 18 imperfect answers are retrieval failures (section 3), which a larger model would not fix: improving retrieval is the better investment.

---
## 5. The saved model or pipeline

Reload what you saved in notebook 02 and use it on a new example, exactly as the app does.

**The saved pipeline, used as the app will use it.** Everything is reloaded from the files saved in notebook 02 (`models/config.json` and the index in `models/chroma/`), and asked a question that is **not** in the evaluation set, about The Witcher 3, the only game without its own evaluation question. As in the app, the user selects a game and types a question; `answer_question` cleans the question with the same `clean_text` used on the reviews, retrieves the 5 closest chunks for that game, and answers from them with citations.

In [20]:
cfg, app_collection, app_embedder, app_client = load_pipeline()

user_game = "The Witcher 3: Wild Hunt"
user_question = "Is The Witcher 3 still worth playing today, or does it feel dated?"

out = answer_question(user_question, app_collection, app_embedder, app_client,
                      k=cfg["k"], game=user_game if cfg["game_filter"] else None)

print(f"Game: {user_game}\nQuestion: {user_question}\n")
print(textwrap.fill(out["answer"], width=110))
print(f"\nCited: {out['cited_ids']} | Invented: {out['invented_citations'] or 'none'} | "
      f"closest distance {out['min_distance']} | ${out['cost_usd']:.5f} | {out['seconds']} s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Game: The Witcher 3: Wild Hunt
Question: Is The Witcher 3 still worth playing today, or does it feel dated?

Based on these Steam reviews, *The Witcher 3: Wild Hunt* appears to still be worth playing today, with players
praising its long-lasting entertainment and strong story even after many hours. [review 194647207] [review
30794628] [review 50387772] [review 139795652] Players also specifically mention that it doesn’t feel too
outdated visually—e.g., one reviewer says the graphics still look like they’re from 2019. [review 50387772]
Evidence is strongest that it remains enjoyable, though opinions on how “dated” it feels aren’t tested in a
consistent way across all reviewers (the only “dated” angle mentioned is that some elements are still “rough
around the edges”). [review 50752020] Also, note that at least one positive assessment attributes improvements
to updates, so impressions could change after patches. [review 139795652]

Cited: 139795652;194647207;30794628;50387772;50752020 | 

In [21]:
sources = retrieve(user_question, app_collection, app_embedder, k=cfg["k"], game=user_game)
sources.assign(verdict=sources["voted_up"].map({True: "👍", False: "👎"}),
               excerpt=sources["text"].str[:150])[
    ["review_id", "verdict", "playtime_h", "created_date", "distance", "excerpt"]].round(3)

,review_id,verdict,playtime_h,created_date,distance,excerpt
0,194647207,👍,247.267,2025-05-11,0.564,The Witcher 3 is a gift.\n\nI'm at a loss for words—this game has provided me with months of entertainment and countless heartwarming moments.\n\nNever ha
1,30794628,👍,29.750,2017-03-28,0.575,"This is the definition of bang for your buck. The Witcher 3 is one of the best games ever made with the most engrossing storyline, epic battles, enter"
2,50752020,👍,323.550,2019-05-21,0.586,"Oh man I love it even more now.\nUpon completing The Witcher 2, or probably even before that, I pre-ordered the shit out of The Witcher 3's collector's"
3,50387772,👍,63.417,2019-04-29,0.588,Witcher 3 has Sat untouched in my game library for almost 3 years. I should have played it sooner... After completing the Witcher 3 all I can say how
4,139795652,👍,290.850,2023-06-09,0.589,"Furthermore, the recent updates have addressed many technical issues and introduced quality-of-life improvements, enhancing the overall gameplay exper"


**Reading the answer.** The saved pipeline works on its own and on a question nobody prepared for: the answer is grounded (5 citations, none invented), balanced without being asked to be ("still worth playing", but "rough around the edges"), and honest about the limits of its evidence. Answering took 2.9 seconds and cost $0.00045.

**But it shows the time-reasoning weakness from section 3.** The question asks about *today*. The claim that the game "doesn't feel too outdated visually" comes from a review written in **2019** (50387772), which cannot say how the game looks in 2026; only one of the five retrieved reviews is recent (2025). The answer presents the 2019 opinion as evidence for today without noticing the gap. The model is never told today's date or when the reviews were collected, so it cannot reason about how old its evidence is: giving it both is a concrete next step.

---
## 6. Conclusions and limitations

Would you ship it, to whom, and with what warning? What data, products or questions does it hold for? What would you do next?

**Would I ship it?** Yes, as a beta assistant for PC players who want **evidence** before buying, not a verdict. On 26 questions it scores **0.63 against 0.33** for the same LLM without reviews, refuses all 4 out-of-scope questions, never invented a fact or a source, and costs about **$11.60 a month** for 1,000 questions a day, at under 2 seconds per answer. Its main gain is exactly where an LLM alone fails: recent games, changes after patches, and specific complaints.

**With what warning?** Shown next to every answer: *"Based on 5 Steam reviews, shown below. It can miss minority opinions, and older reviews may not describe the game today."* The sources table under each answer is part of the product, not decoration: it is what lets a player check the answer.

**What it holds for:** questions about how players received the 23 games in the corpus, from English Steam reviews (so the PC version) collected on 10 November 2026. **What it does not hold for:** prices, release dates, consoles, other games (it refuses these correctly), and, with caution, "is there anything wrong with it?" questions about well-liked games, specific angles such as the late game, brand-new content, and "is it still…?" questions (section 3).

**How far to trust these results:**
- **A small evaluation:** 30 questions, graded by one person (me), one run per question. The same question can get different answers on two runs (notebook 02, R2), so differences of a few points are not reliable, and per-type scores rest on 2–11 questions.
- **The hit rate is approximate:** patterns miss paraphrases (F2) and can match negations, so it measures on-topic evidence found, not answers.
- **The evaluation set shaped one decision:** the game filter was compared on it, so that comparison is not fully independent. And every evaluation question names its game, which is why the filter made no difference here, though real users will not always name it.
- **The data has blind spots:** the Helpful sort under-represents the newest reviews (only 16 chunks discuss the DLC released 12 days before collection); Steam keeps only the latest version of edited reviews (18.5%); bilingual reviews and other languages were removed.

**What I would do next, in order of expected impact:**
1. **Improve retrieval, which causes 12 of the 18 imperfect answers:** hybrid search (keywords + embeddings) for names the embedding model does not know, retrieving more chunks and re-ranking them, and rewriting the question to capture its angle ("complaints about…" → negative reviews).
2. **Tell the model today's date and the collection date**, so it can tell old evidence from current evidence (sections 3 and 5).
3. **Refuse when the closest chunk is far** (R2's distance was 0.76 against 0.34–0.66 elsewhere), with the threshold chosen on new questions, not on this evaluation set.
4. **Let users select two games** for comparisons.
5. **Evaluate more:** more questions, a second grader, and several runs per question to measure variability.